## Definitive regime study \u2014 HG-kNN vs HG-Matmul vs HG-ANN, time vs n\n
Sweeps `n` for a **low-d** (d=2) and a **very-high-d** (d=512) synthetic regime and times each backend's **step-2 (k-NN)** \u2014 the only part that differs (step 3 is identical), so no Mt-KaHyPar is needed and it runs anywhere. Produces a two-panel time-vs-n log-log plot plus a confirmation table. The three regime winners fall out and map to your three cases: **(small n, low d) \u2192 tree**, **(small n, high d) \u2192 matmul**, **(large n, high d) \u2192 ANN**. High-d rows get heavy on CPU at large n (the tree especially) \u2014 a GPU (torch) speeds the matmul row; trim/extend `N_HIGH`/`N_LOW` as needed. Requires numpy, scikit-learn, hnswlib.\n

In [1]:
"""
Definitive regime study: HG-kNN vs HG-Matmul vs HG-ANN, step-2 (k-NN) time vs n, on synthetic
data, across the low-d and high-d regimes. No Mt-KaHyPar needed -- step 2 is the only part that
differs between backends (step 3 is identical), so timing step 2 isolates the comparison and the
notebook runs anywhere.

Backends (all EXACT-equivalent neighbours except ANN, which is high-recall approximate):
  HG-kNN     -- KD-tree (Euclidean). Fast in low d, degrades toward n^2 in high d (curse of dim).
  HG-Matmul  -- dense distance matmul (BLAS/GPU). Exact, O(n^2 d), hardware-optimal.
  HG-ANN     -- HNSW (l2) + exact rerank. ~O(n log n), worse constant; wins only at high d, large n.

Requires: numpy, scikit-learn, hnswlib (+ optional torch for the matmul GPU path).
"""
import time, numpy as np

# ----------------------------- CONFIG -----------------------------
D_LOW,  N_LOW  = 2,   [1000, 10000, 100000]        # low-d is cheap -> sweep big
D_HIGH, N_HIGH = 512, [1000, 3000, 10000, 30000]   # very-high-d: tree/matmul heavy -> moderate n (GPU can push further)
K_NN     = 15
N_GROUPS = 50
SEP      = 10.0
SPREAD_FRAC = 0.03
SEED     = 42
USE_TORCH = True          # matmul GPU/CPU via torch if available; numpy fallback otherwise
ANN_RERANK_MULT, ANN_EF, ANN_M = 3, 100, 16

# ------------------- synthetic data (d-invariant separation) ------
def _maximin_centers(n_groups, d, sep, rng, pool_mult=60):
    box = sep * (n_groups ** (1.0 / d)) * 1.5
    pool = rng.uniform(0.0, box, size=(max(n_groups * pool_mult, n_groups), d))
    first = int(rng.integers(len(pool))); centers = [pool[first]]
    md2 = np.sum((pool - pool[first]) ** 2, axis=1)
    for _ in range(1, n_groups):
        i = int(np.argmax(md2)); centers.append(pool[i]); md2 = np.minimum(md2, np.sum((pool - pool[i]) ** 2, axis=1))
    centers = np.asarray(centers, float)
    diff = centers[:, None, :] - centers[None, :, :]; dist = np.sqrt((diff ** 2).sum(-1)); dist[np.diag_indices(n_groups)] = np.inf
    centers *= sep / dist.min(); return centers

def make_blobs(n, d, n_groups=N_GROUPS, sep=SEP, spread_frac=SPREAD_FRAC, seed=SEED):
    rng = np.random.default_rng(seed); C = _maximin_centers(n_groups, d, sep, rng); std = spread_frac * sep / np.sqrt(d)
    sizes = np.full(n_groups, n // n_groups, int); sizes[: n % n_groups] += 1
    X = np.empty((n, d)); pos = 0
    for g, m in enumerate(sizes): X[pos:pos + m] = C[g] + rng.normal(0, std, size=(m, d)); pos += m
    return np.ascontiguousarray(X[rng.permutation(n)], dtype=np.float32)

# --------------------------- backends -----------------------------
def knn_tree(X, k):
    from sklearn.neighbors import NearestNeighbors
    NearestNeighbors(n_neighbors=k + 1, algorithm="kd_tree", metric="euclidean", n_jobs=-1).fit(X).kneighbors(X)

def knn_matmul(X, k, block=4096, use_torch=USE_TORCH):
    if use_torch:
        try:
            import torch
            dev = "cuda" if torch.cuda.is_available() else "cpu"; Xt = torch.as_tensor(X, device=dev)
            sq = (Xt * Xt).sum(1); n = Xt.shape[0]
            for s in range(0, n, block):
                e = min(s + block, n); d2 = sq[s:e, None] + sq[None, :] - 2.0 * (Xt[s:e] @ Xt.T)
                torch.topk(d2, k + 1, dim=1, largest=False)
            if dev == "cuda": torch.cuda.synchronize()
            return
        except Exception:
            pass
    sq = (X * X).sum(1); n = X.shape[0]
    for s in range(0, n, block):
        e = min(s + block, n); d2 = sq[s:e, None] + sq[None, :] - 2.0 * (X[s:e] @ X.T); np.maximum(d2, 0, out=d2)
        np.argpartition(d2, k, axis=1)[:, :k]

def knn_ann(X, k, rerank_mult=ANN_RERANK_MULT, ef=ANN_EF, M=ANN_M):
    import hnswlib
    n, d = X.shape; kk = min(n, k * rerank_mult + 1)
    ix = hnswlib.Index(space="l2", dim=d); ix.init_index(max_elements=n, ef_construction=200, M=M)
    ix.add_items(X, np.arange(n), num_threads=-1); ix.set_ef(max(ef, kk))
    cand, _ = ix.knn_query(X, k=kk, num_threads=-1)
    for s in range(0, n, 512):
        e = min(s + 512, n); c = cand[s:e]; diff = X[s:e][:, None, :] - X[c]; d2 = (diff ** 2).sum(-1)
        d2[c == np.arange(s, e)[:, None]] = np.inf; np.argsort(d2, axis=1)[:, :k]

BACKENDS = [("HG-kNN", knn_tree), ("HG-Matmul", knn_matmul), ("HG-ANN", knn_ann)]

def sweep(d, ns, k=K_NN):
    out = {name: [] for name, _ in BACKENDS}
    for n in ns:
        X = make_blobs(n, d)
        for name, fn in BACKENDS:
            t = time.perf_counter(); fn(X, k); out[name].append(time.perf_counter() - t)
        print(f"  d={d:4d} n={n:>7}: " + "  ".join(f"{nm}={out[nm][-1]:7.3f}s" for nm, _ in BACKENDS))
    return out


# --------------------------- warmup -------------------------------
def _warmup():
    Xw = make_blobs(200, 4)
    for _, fn in BACKENDS:
        try: fn(Xw, K_NN)
        except Exception: pass

# ----------------------- plot + confirm ---------------------------
_COL = {"HG-kNN": "#27ae60", "HG-Matmul": "#c0392b", "HG-ANN": "#2980b9"}
_MK  = {"HG-kNN": "^", "HG-Matmul": "o", "HG-ANN": "s"}

def plot_regimes(results, out_path="regime_scaling.png"):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(results), figsize=(6.6 * len(results), 5.4), squeeze=False)
    for ax, (label, d, ns, data) in zip(axes[0], results):
        n = np.asarray(ns, float)
        for name in ["HG-Matmul", "HG-kNN", "HG-ANN"]:
            ax.loglog(n, data[name], _MK[name] + "-", color=_COL[name], lw=2, ms=7, label=name)
        x = np.array([n[0], n[-1]])
        ax.loglog(x, data["HG-Matmul"][0] * (x / n[0]) ** 2, "--", color="#c0392b", alpha=.35, lw=1, label="n\u00b2 ref")
        ax.loglog(x, data["HG-kNN"][0] * (x / n[0]) ** 1, "--", color="#27ae60", alpha=.35, lw=1, label="n ref")
        ax.set_xlabel("n (records)"); ax.set_ylabel("step-2 k-NN time (s)")
        ax.set_title(f"{label} (d={d})"); ax.grid(True, which="both", alpha=.25); ax.legend(fontsize=8, loc="upper left")
    fig.suptitle("HG step-2 scaling by regime: low-d -> tree; very-high-d -> matmul (small n) then ANN (large n)", fontsize=11)
    fig.tight_layout(); fig.savefig(out_path, dpi=120); print(f"saved {out_path}")

def confirm(results):
    pred = {("low-d", "small"): "HG-kNN",   ("low-d", "large"): "HG-kNN",
            ("high-d", "small"): "HG-Matmul", ("high-d", "large"): "HG-ANN"}
    print("\n" + "=" * 76)
    print("  REGIME CONFIRMATION: empirical fastest vs theory prediction")
    print("=" * 76)
    print(f"  {'regime':<9}{'d':>6}{'n':>9}{'empirical':>13}{'predicted':>13}{'match':>9}")
    print("-" * 76)
    for label, d, ns, data in results:
        for pos, idx in [("small", 0), ("large", len(ns) - 1)]:
            emp = min(("HG-kNN", "HG-Matmul", "HG-ANN"), key=lambda nm: data[nm][idx])
            p = pred[(label, pos)]
            print(f"  {label:<9}{d:>6}{ns[idx]:>9}{emp:>13}{p:>13}{'OK' if emp == p else 'DIFF':>9}")
    print("-" * 76)
    for label, d, ns, data in results:
        ex = ", ".join(f"{nm} {np.polyfit(np.log10(ns), np.log10(data[nm]), 1)[0]:.2f}" for nm in ["HG-kNN","HG-Matmul","HG-ANN"])
        print(f"  {label} (d={d}) exponents: {ex}")
    print("=" * 76)
    print("  Maps to the three cases: (small n, low d)->tree; (small n, high d)->matmul; (large n, high d)->ANN.")
    print("=" * 76 + "\n")

def main():
    print(f"Regime study: low-d={D_LOW}{N_LOW}, high-d={D_HIGH}{N_HIGH}, k={K_NN}")
    _warmup()
    results = []
    for label, d, ns in [("low-d", D_LOW, N_LOW), ("high-d", D_HIGH, N_HIGH)]:
        print(f"\n--- {label} regime, d={d} ---")
        results.append((label, d, ns, sweep(d, ns)))
    plot_regimes(results, out_path="regime_scaling.png")
    confirm(results)

if __name__ == "__main__":
    main()


Regime study: low-d=2[1000, 10000, 100000], high-d=512[1000, 3000, 10000, 30000], k=15

--- low-d regime, d=2 ---
  d=   2 n=   1000: HG-kNN=  0.012s  HG-Matmul=  0.006s  HG-ANN=  0.013s
  d=   2 n=  10000: HG-kNN=  0.016s  HG-Matmul=  0.266s  HG-ANN=  0.121s
  d=   2 n= 100000: HG-kNN=  0.099s  HG-Matmul= 21.420s  HG-ANN=  1.259s

--- high-d regime, d=512 ---
  d= 512 n=   1000: HG-kNN=  0.034s  HG-Matmul=  0.005s  HG-ANN=  0.141s
  d= 512 n=   3000: HG-kNN=  0.077s  HG-Matmul=  0.026s  HG-ANN=  0.441s
  d= 512 n=  10000: HG-kNN=  0.493s  HG-Matmul=  0.247s  HG-ANN=  1.557s
  d= 512 n=  30000: HG-kNN=  3.187s  HG-Matmul=  2.118s  HG-ANN=  5.375s
saved regime_scaling.png

  REGIME CONFIRMATION: empirical fastest vs theory prediction
  regime        d        n    empirical    predicted    match
----------------------------------------------------------------------------
  low-d         2     1000    HG-Matmul       HG-kNN     DIFF
  low-d         2   100000       HG-kNN       HG-kNN    